# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areebaarain/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
### 1. Method choice and why

I chose a **baseline scoring model using a Decision Tree classifier**. This method fits my Refresh / Content Opportunity Scoring lane because the goal is to identify pages that may need action based on their search performance patterns.

A Decision Tree is a good starting method because it is easy to understand and explain. It can learn simple patterns from features such as search volume, position, clicks, impressions, or performance trends and classify pages into possible action groups.

This fits the lane because the final goal is not just to predict a number, but to support a decision: **which pages should be refreshed or reviewed first**. The model provides an interpretable baseline that can later be compared with more advanced methods.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
## 2. Split design

I used a **time-aware train/test split**. Earlier observations are used for training, while later observations are kept for testing.

This is more honest for my Refresh / Content Opportunity Scoring question because, in a real workflow, we would use past search performance to decide which pages should be refreshed or reviewed in the future. A random split could mix earlier and later observations and allow information from the future to influence the training data.

The time-aware split therefore better represents the real decision process and reduces the risk of temporal leakage.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
## 3. Train + compare vs my baseline

I trained a Decision Tree classifier using the same dataset and the same 80/20 train-test split for both models. I compared it with my Week-4 baseline using accuracy, weighted precision, weighted recall, and weighted F1 score.

The Week-4 baseline achieved an accuracy of 0.601 and a weighted F1 score of 0.451. The Decision Tree improved the accuracy to 0.639 and the weighted F1 score to 0.527.

The Decision Tree performed better than the baseline across all comparison metrics. The largest improvement was in weighted precision, which increased from 0.361 to 0.568. This suggests that the model learned useful patterns from the content and search-performance features instead of relying only on the baseline prediction.


In [7]:
!git clone https://github.com/Areebaarain/flyrank-ml-internship.git


fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [8]:
import pandas as pd
import os

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

File exists: True
Rows: 30000
Columns: 44


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# 3. Train + compare vs my baseline
# Same data, same split, same metric
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --------------------------------------------
# 1. Copy the data
# --------------------------------------------

model_df = df.copy()

# --------------------------------------------
# 2. Choose target
# --------------------------------------------

target = "trend_direction"

# Features available BEFORE making the decision
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

# Keep only required columns
model_df = model_df[features + [target]].dropna()

# --------------------------------------------
# 3. Encode categorical target
# --------------------------------------------

label_encoder = LabelEncoder()
model_df[target] = label_encoder.fit_transform(model_df[target])

X = model_df[features]
y = model_df[target]

# --------------------------------------------
# 4. SAME fixed split for both models
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================
# BASELINE MODEL
# Majority-class prediction
# ============================================

majority_class = y_train.mode()[0]

baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=majority_class
)

# ============================================
# DECISION TREE MODEL
# ============================================

tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

# ============================================
# SAME METRICS FOR BOTH
# ============================================

results = pd.DataFrame({
    "Model": [
        "Week-4 Baseline (Majority Class)",
        "Decision Tree (Depth 4)"
    ],

    "Accuracy": [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, tree_predictions)
    ],

    "Precision (weighted)": [
        precision_score(y_test, baseline_predictions, average="weighted", zero_division=0),
        precision_score(y_test, tree_predictions, average="weighted", zero_division=0)
    ],

    "Recall (weighted)": [
        recall_score(y_test, baseline_predictions, average="weighted", zero_division=0),
        recall_score(y_test, tree_predictions, average="weighted", zero_division=0)
    ],

    "F1 Score (weighted)": [
        f1_score(y_test, baseline_predictions, average="weighted", zero_division=0),
        f1_score(y_test, tree_predictions, average="weighted", zero_division=0)
    ]
})

# Round results
results = results.round(3)

results


,Model,Accuracy,Precision (weighted),Recall (weighted),F1 Score (weighted)
0,Week-4 Baseline (Majority Class),0.601,0.361,0.601,0.451
1,Decision Tree (Depth 4),0.639,0.568,0.639,0.527


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# 4. Errors and interpretation
# ============================================

from sklearn.metrics import confusion_matrix, classification_report

# --------------------------------------------
# 1. Find wrong predictions
# --------------------------------------------

error_df = X_test.copy()

error_df["Actual"] = y_test.values
error_df["Predicted"] = tree_predictions

errors = error_df[
    error_df["Actual"] != error_df["Predicted"]
].copy()

print("Total test examples:", len(error_df))
print("Wrong predictions:", len(errors))
print("Error rate:", round(len(errors) / len(error_df), 3))

print("\nSample wrong predictions:")
display(errors.head(10))


# --------------------------------------------
# 2. Confusion matrix
# --------------------------------------------

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tree_predictions))


# --------------------------------------------
# 3. Feature importance
# --------------------------------------------

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTop features the model relies on:")
display(feature_importance.head(8))


Total test examples: 3980
Wrong predictions: 1438
Error rate: 0.361

Sample wrong predictions:


,search_volume,competition,cpc,word_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,Actual,Predicted
28002,0.0,0.00,0.00,2321.0,1335,1,8,7,7,2,517,11,0.07,7.6,28.57,37.50,3,0
10854,10.0,0.00,0.00,5599.0,1078,1,37,20,20,1,236,104,0.09,33.5,5.00,0.00,3,0
6854,0.0,0.00,0.00,3294.0,3059,39,184,107,104,5,124,104,1.27,14.3,4.67,4.35,4,0
6672,0.0,0.00,0.00,2379.0,94,0,11,11,11,0,133,20,0.00,42.0,0.00,0.00,4,0
2248,140.0,0.03,0.03,2806.0,15138,62,72,60,60,2,124,20,0.41,5.3,3.33,2.78,3,0
10685,110.0,0.11,0.92,2592.0,2779,3,16,15,15,0,545,13,0.11,14.0,0.00,0.00,4,0
20229,10.0,0.11,0.00,3036.0,24111,27,64,60,56,0,362,20,0.11,8.6,0.00,1.56,3,0
4201,10.0,0.00,0.00,1392.0,484,1,2,2,2,0,111,20,0.21,10.5,0.00,50.00,3,0
29131,20.0,0.02,0.00,2547.0,5355,23,46,40,38,2,119,20,0.43,10.8,5.00,4.35,4,0
25168,0.0,0.00,0.00,3021.0,8,0,1,1,1,0,145,20,0.00,28.1,0.00,0.00,3,0



Confusion Matrix:
[[2343   36    3    9    0]
 [  74   96    5    0    0]
 [ 112   25   84    0    0]
 [ 669    7    0   19    1]
 [ 490    5    0    2    0]]

Top features the model relies on:


,Feature,Importance
4,impressions_90d,0.544663
13,avg_position,0.170389
5,clicks_90d,0.124131
10,content_age_days,0.077316
15,scroll_rate,0.061443
11,days_since_last_update,0.018861
6,pageviews_90d,0.003197
0,search_volume,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.